In [6]:
import os

import pandas as pd
import numpy as np
import xarray as xr

import matplotlib.pyplot as plt

from bluemath_tk.core.io import load_model

In [4]:
#### Load Data
pcs_df_with_kma = pd.read_csv(os.path.join("..", "..", "outputs", "alr", "pcs_with_kma.csv"))
pcs_df_with_kma = pcs_df_with_kma.rename(columns={'kma':'bmus', 'Unnamed: 0':'time'})

pcs_df_with_kma = pcs_df_with_kma.set_index('time')
pcs_df_with_kma.index = pd.to_datetime(pcs_df_with_kma.index)

In [5]:
import numpy as np
from bluemath_tk.teslakit.statistical import CopulaSimulation

calculated_PCs = {}
simulated_PCs = {}

num_clusters = pcs_df_with_kma.bmus.max() + 1
pc_cols = ["PC1", "PC2", "PC3", "PC4", "PC5", "PC6"]

for cluster in range(0, num_clusters):

    calculated_PCs[f"{cluster}"] = pcs_df_with_kma.loc[pcs_df_with_kma.bmus == cluster, pc_cols].values

    simulated_PCs[f"{cluster}"] = CopulaSimulation(
        U_data=calculated_PCs[f"{cluster}"],
        kernels=["KDE"] * len(pc_cols),
        num_sim=100000,
    )

In [7]:
kma_ob = load_model('/lustre/geocean/WORK/users/alonsoap/projects/TWINPEACS/02_Emulator_Testing/wt_var_testing/outputs/alr/classifier.pkl')

In [8]:
for k, samples in simulated_PCs.items():
    lab, _ = kma_ob.predict(pd.DataFrame(samples, columns=pc_cols))
    simulated_PCs[k] = samples[lab.kma_bmus.values == int(k)]

In [9]:
simulation = xr.open_dataset('/lustre/geocean/WORK/users/alonsoap/projects/TWINPEACS/02_Emulator_Testing/wt_var_testing/outputs/alr/wt_alr_sim_bmus.nc')
bmus_sims  = simulation.evbmus_sims.transpose("n_sim", "time").values.astype(int)   # (n_sim, n_days)
n_sim, n_days = bmus_sims.shape

rng = np.random.default_rng(0)
all_simulated_PCs = np.zeros((n_sim, n_days, len(pc_cols)))

for s in range(n_sim):
    for wt in np.unique(bmus_sims[s]):
        days = np.flatnonzero(bmus_sims[s] == wt)            # days of this record in type wt
        pool = simulated_PCs[f"{wt}"]                        # KDE samples of type wt
        all_simulated_PCs[s, days] = pool[rng.integers(len(pool), size=len(days))]

sim = xr.Dataset(
    {"pcs":  (("seed", "time", "pc"), all_simulated_PCs),
     "bmus": (("seed", "time"), bmus_sims)},
    coords={"seed": range(n_sim), "time": simulation.time.values.astype("datetime64[ns]"), "pc": pc_cols},
)
sim

<xarray.Dataset> Size: 178MB
Dimensions:  (seed: 100, time: 31685, pc: 6)
Coordinates:
  * seed     (seed) int64 800B 0 1 2 3 4 5 6 7 8 ... 91 92 93 94 95 96 97 98 99
  * time     (time) datetime64[ns] 253kB 1940-01-01 1940-01-02 ... 2026-09-30
  * pc       (pc) <U3 72B 'PC1' 'PC2' 'PC3' 'PC4' 'PC5' 'PC6'
Data variables:
    pcs      (seed, time, pc) float64 152MB 52.84 32.06 ... -0.4246 -5.249
    bmus     (seed, time) int64 25MB 31 31 29 29 5 33 20 ... 32 19 19 19 3 9 9

In [10]:
sim.to_netcdf("../../outputs/model_comp/alr_kde_filter.nc")

In [11]:
results = []
pc_cols = ['PC1','PC2','PC3','PC4','PC5','PC6']

for s in sim.seed.values:
    sim_df = sim.isel(seed=s)['pcs'].to_pandas()
    kma_predict, _ = kma_ob.predict(sim_df[pc_cols])

    sim_df['bmus_alr']     = sim.isel(seed=s).bmus.values
    sim_df['bmus_reclass'] = kma_predict.kma_bmus.values
    sim_df['seed']         = s
    results.append(sim_df[['seed', 'bmus_alr', 'bmus_reclass']])

check = pd.concat(results)
check['match'] = check.bmus_alr == check.bmus_reclass

print(f"overall: {check.match.mean():.1%}")

overall: 100.0%
